# Day 10 · Qwen3-VL-4B grounding · Kaggle GPU

Run cells in order. Attach the Kaggle road-damage dataset with **Add Input**, enable a **T4 GPU** and Internet so Hugging Face weights can download. The notebook keeps data, predictions and weights under ignored `data/` and `work/`. It produces real `report/evidence/*.json` by calling the examiner's `scripts/score.py`.

E0 is zero-shot. E1–E3 are 300-step SFT; E4 changes only train data size; E5 is the GRPO bonus. E5 may need more VRAM than SFT. No test score is entered manually.


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = "https://github.com/vminhquan/K4-Track-4-Day10-VoMinhQuan-2A202602429-VisionLanguageModel.git"
ROOT = Path("/kaggle/working/day10")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
os.chdir(ROOT)
print("Working in", Path.cwd())

def run(*args):
    command = [sys.executable, *map(str, args)]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)


## 0. Install and validate the examiner tools

Run after enabling GPU. Package installation may require a kernel restart on Kaggle. Rerun this notebook from the first cell after a restart; `data/` and completed checkpoints remain in `/kaggle/working` for the current session.


In [ ]:
run("-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "pytest")
run("-m", "pytest", "-q")
run("-m", "unittest", "-v", "test_bonus_reward.py")
run("-m", "unittest", "-v", "test_student_workflow.py")


## 1. Prepare the fixed session split

Attach `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes` via **Add Input**. The examiner's script finds its `images/` and `labels-YOLO/` folder under `/kaggle/input`.


In [ ]:
if not Path("data/splits/meta.json").exists():
    run("scripts/prepare_data.py", "--source", "/kaggle/input")
run("lab_workflow.py", "check-data")
run("lab_workflow.py", "gt-figures")
run("lab_workflow.py", "preflight")  # model, 220 visual tokens, answer-only mask


## 2. E0 zero-shot on the full 311-image test split

Save the raw predictions in `work/e0_test.jsonl`; the scoring script writes the evidence JSON.


In [ ]:
run("lab_workflow.py", "predict", "--name", "e0")
run("lab_workflow.py", "score", "--name", "e0")


## 3. E1: LoRA on the LLM only

A fresh Python process runs each stage, so VRAM is released before evaluation. Each SFT run saves an adapter and `run.json` every 100 steps. If Kaggle reports OOM, use `--batch-size 1 --grad-accum 8` in the train call; effective batch remains 8.


In [ ]:
run("lab_workflow.py", "train", "--name", "e1")
run("lab_workflow.py", "predict", "--name", "e1")
run("lab_workflow.py", "score", "--name", "e1")
# Diagnostic for REPORT C3: deliberately score normalized predictions as pixels.
run("scripts/score.py", "--pred", "work/e1_test.jsonl", "--name", "E1_PIXEL_MISMATCH",
    "--coord", "pixel", "--out", "report/evidence/diagnostic_coord_pixel.json")


## 4. E2: LLM LoRA + full float32 projector

The projector is saved separately as `work/e2/projector.pt` and reloaded for inference.


In [ ]:
run("lab_workflow.py", "train", "--name", "e2")
run("lab_workflow.py", "predict", "--name", "e2")
run("lab_workflow.py", "score", "--name", "e2")


## 5. E3: E2 + ViT LoRA

Expected LoRA target counts: E1/E2 = 252, E3 = 348. The code stops if the model module tree differs.


In [ ]:
run("lab_workflow.py", "train", "--name", "e3")
run("lab_workflow.py", "predict", "--name", "e3")
run("lab_workflow.py", "score", "--name", "e3")
run("lab_workflow.py", "results")


## 6. E4: clean ablation of train data size

Pick the best E1–E3 by test F1. The 100% condition reuses its existing checkpoint and reruns inference/scoring; the 10% condition retrains from the same base model with every other setting unchanged. The fixed 10% sample uses seed 42. Both evidence files are generated by `score.py`.


In [ ]:
best = subprocess.check_output([sys.executable, "lab_workflow.py", "best"], text=True).strip()
print("Best SFT setup:", best)
run("lab_workflow.py", "predict", "--name", "e4_data_1.0", "--base-setup", best)
run("lab_workflow.py", "score", "--name", "e4_data_1.0", "--base-setup", best)
run("lab_workflow.py", "train", "--name", "e4_data_0.1", "--base-setup", best, "--fraction", "0.1")
run("lab_workflow.py", "predict", "--name", "e4_data_0.1")
run("lab_workflow.py", "score", "--name", "e4_data_0.1")
run("lab_workflow.py", "results")


## 7. Bonus E5: GRPO with verifiable IoU reward

The reward has six independent CPU tests (including all five required cases). GRPO starts from the best SFT adapter, generates four completions per image, trains for 60 steps at 5e-6 and saves its own adapter/projector and log. `trl==1.9.2` supports VLM data, but Qwen3-VL compatibility and Kaggle GPU memory must be checked in the actual run.


In [ ]:
run("-m", "unittest", "-v", "test_bonus_reward.py")
run("lab_workflow.py", "grpo")
run("lab_workflow.py", "predict", "--name", "e5")
run("lab_workflow.py", "score", "--name", "e5")
run("lab_workflow.py", "results")


## 8. Visual evidence and return package

Create six paired GT/prediction figures from the best SFT model. Download the ZIP and send it back for the final `REPORT.md` analysis. The ZIP excludes images, model weights and API tokens.


In [ ]:
best = subprocess.check_output([sys.executable, "lab_workflow.py", "best"], text=True).strip()
run("lab_workflow.py", "figures", "--name", best)
subprocess.run([sys.executable, "scripts/check_submission.py"], check=False)
print("REPORT.md still needs the measured results and analysis before final submission.")


In [ ]:
import zipfile
archive = Path("/kaggle/working/day10_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ("report/evidence/*.json", "report/figures/*.png",
                    "work/*/run.json", "work/*_test.jsonl", "data/splits/meta.json"):
        for path in Path(".").glob(pattern):
            z.write(path, path.as_posix())
print("Download:", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")
from IPython.display import FileLink, display
display(FileLink(str(archive)))
